# 📅 2026-09-03 개발 노트 : 배치 장애 복구 + 학생 모델 캘리브레이션 + 운영 크래시 수습 + 홍보 전 UX 정비

## 🎯 오늘의 목표 — "백필 시작 가능한 상태로"

- [x] OpenAI Batch API 장애 원인 격리 → 새 프로젝트 키로 해결
- [x] 백필 리허설 100개 (크롤러 429/조기종료 개선)
- [x] 학생 모델 캘리브레이션 실험 → gpt-5.4-mini + 12-shot 확정
- [x] 스팀 로그인 운영 크래시(openid 누락) 수습, Railway 11시간 다운 복구
- [x] README 시니어 톤으로 전면 재작성 + .env.example
- [x] 홍보 전 UX 정비: 유명작 필터, 참조 게임 라우팅, 프리셋, 카드 지표, 상세 히어로
- [ ] 100개 재분석(5.4-mini) → 적재 → 임베딩 → 백분위 (다음)
- [ ] 백필 `--limit 500` 반복 + 주간 스케줄 등록 (다음)

> 오늘 배운 한 줄: **"인증은 되는데 특정 기능만 실패"는 최소 재현 + 다른 클라이언트로 계층을 격리하면 빠르다.**

## ⭐ 1. OpenAI Batch "Cannot find file" — 조직 레벨 장애

**문제:** 배치가 전부 `Cannot find file file-xxx, or organization org-... does not have access` 로 실패.
파일은 업로드·processed·`files.retrieve` 조회까지 정상인데 배치 검증만 실패하는 모순.

**격리 과정 (가설 → 기각):**
1. 업로드-생성 레이스? → `processed` 폴링 추가 후 재시도 → **같은 에러** (기각)
2. 파일 내용/크기(2.7MB, few-shot 포함)? → **173바이트 1건짜리 스모크 배치**(`embeddings/batch_smoke_test.py`) → 같은 에러 (기각)
3. 키 권한? → 대시보드 확인: Permissions **All** (기각)
4. 코드/API 경로? → **웹 대시보드에서 직접 만든 배치도 동일 실패** → 조직/프로젝트 레벨 확정

**해결:** 새 프로젝트 생성 → 새 키 → `.env` 교체 → `docker compose up -d --force-recreate batch`
(env_file은 재생성해야 반영) → 스모크 `in_progress` 진입 확인. 원인 불명이지만 프로젝트 단위 플래그로 추정.

**장애 중 가동 유지:** `batch_generator --sync` 동기 폴백 모드 추가. 출력 포맷을 배치와 동일하게 맞춰
batch_processor 등 후속 단계 무수정. `weekly_pipeline --sync` 패스스루.

## 2. 백필 리허설 — 크롤러 개선 3건

**429 폭격:** 백필은 후보 2만 개 = 400페이지를 넘기는데 페이지 간격 1.0s → 스팀 스토어 검색 429.
백오프는 있었지만 성공하면 다시 1s로 돌아가 반복 429.
→ 기본 간격 1.6s + **적응형 지연**(429마다 +0.5s, 최대 5s). 한 번 걸리면 알아서 순해짐.

**15분 침묵:** 페이징 중 로그가 0줄이라 멈춘 줄 알았음 → 20페이지마다 진행 로그.

**불필요한 페이징:** 출시일 역순인데 무조건 2만 개를 돌았음 → 검색 결과 HTML의 `search_released`를
파싱해 **시작일 이전 구간 도달 시 조기 종료**. 실행당 10분+ 절약.

**결과:** 100개 수집 정상 (9/1~9/2 신작, 중복 1, 기간 밖 0), `is_active=FALSE pending` 등록.
CSV 품질: 설명 중앙값 194자, 실제 게임만. 배치 요청 구조도 검증(시스템 8,813자 + few-shot 6쌍 + json_object).

**버그 2개 발견·수정:** ① 신작 0개인데 이전 `new_games.csv`로 배치 제출됨 → 크롤 전 CSV 삭제
② 배치 실패 시 원인 미출력 → `batch.errors` + error file 출력.

## ⭐ 3. 배치 straggler → 부분 수거 도구

100건 배치가 **76/100에서 수 시간 정지** (Batch API 특성, 24h 윈도우).
→ 취소하면 완료분은 output에 보존되는 동작을 이용:
- `embeddings/collect_batch.py`: `--cancel`로 취소 후(cancelling 최대 10분) 완료분 다운로드
- `embeddings/make_retry_csv.py`: `new_games.csv` ∖ `game_metrics` 적재분 = 잔여 CSV
- `wait_and_download`: expired/cancelled여도 output_file_id 있으면 완료분 이어서 적재

교훈: 외부 비동기 시스템은 부분 성공이 기본값. 파이프라인은 전량 성공 아니라 **부분 수거 + 재시도**를 전제로.

## ⭐ 4. 학생 모델 캘리브레이션 드리프트 — 적재 직전 차단

수거한 76건 검증(파싱 76/76, 60지표+9태그 완전체 76/76, 개별 정합성 우수:
도시경영 strategy 8/reflex 2, 탄막 reflex 9). **그런데 gem_potential 분포가 통째로 하향:**

| | 평균 | 최대 | confidence |
|---|---|---|---|
| 교사 GPT-5.4 (4,190) | 75.7 | 100 | - |
| 학생 gpt-4o-mini + 6shot (76) | **28.3** | **40** | 0.68~0.69 균일 |

그대로 적재 → percentile 재계산에서 신작 전원 최하위 → 추천 영구 배제. **적재 보류.**

**A/B 실험 (동일 10게임, sync):** gpt-5.4-mini + 12-shot
→ 대작(Blood of Dawnwalker) **78** / 준수한 인디 **72** / 양산형 **28**, confidence 0.72~0.86 분산.
평균 49.3 — 교사 75.7과의 격차는 코퍼스 차이(교사=선별, 백필=무선별)로 정상.
**변별력이 핵심.** 4o-mini는 전부 25~40에 뭉갬 + confidence 판박이 = 추론 안 함.

**확정:** `weekly_pipeline` 기본값 `gpt-5.4-mini`, `fewshot-n 12`. 76건 폐기, 100개 재분석 예정.
교훈: LLM 품질 게이트는 "파싱 성공"이 아니라 **분포 비교**. 개별 출력이 그럴듯해도 집단 통계가 어긋나면 시스템이 무너진다.

## 5. 스팀 로그인 운영 크래시 — openid 앱 누락

**증상:** 로컬 django 컨테이너 크래시 루프. Railway도 확인해보니 **11시간 전 배포부터 CRASHED**.
로그: `ImproperlyConfigured: The steam provider requires 'allauth.socialaccount.providers.openid'`

**원인:** allauth steam은 openid provider 앱을 INSTALLED_APPS에 **함께** 요구. python3-openid만 넣고 앱은 빠뜨림.
어제 푸시에 수정이 안 들어가서 운영이 그대로 죽어 있었음 (알림이 없어서 11시간 모름 — Railway 배포 실패 알림 켜야 함).

**해결:** `providers.openid` 추가 → 로컬 `compose restart` → Railway 푸시로 복구.
운영 `setup_oauth.py` 실행 시 **Site가 localhost로 덮이는 사고** → `SITE_DOMAIN` env 지원 추가, 운영 재실행으로 복구.
Railway 웹 콘솔은 붙여넣기가 안 돼서 긴 명령 대신 env 변수 + 짧은 스크립트 실행으로 우회.

**확인:** Steam SocialApp 운영 등록 완료, Google은 기존 DB 등록분 유지.

## 6. 홍보 전 UX 정비 (스크린샷 리뷰 기반)

**정체성 정합 — 유명작 필터:** 메인 '오늘의 추천'에 위처·디스코 엘리시움, 랭킹 1위 언더테일 → "숨은 명작" 브랜드와 충돌.
`recommend/by-preference`에 `max_review_count` 추가(스키마·캐시 키·쿼리), 메인/랭킹에서 리뷰 2만 초과 제외.
취향 분석 결과엔 미적용(유저가 고른 조건엔 유명작이 나와도 정당).

**참조 게임 라우팅:** "위쳐같은게임" 검색 → 위쳐 1·2·3이 상위. 임베딩 검색의 구조적 실패(참조 게임 설명이 가장 가까움).
→ 쿼리 분석 GPT에 `reference_game` 필드 추가(비용 0) → DB 매칭 시 **기준 게임 제외하는 by-game 앵커 경로**로 라우팅
+ 프랜차이즈 토큰("witcher") 공유 제외 + 미매칭 시 폴백. 헤더 문구 "X와 비슷한 게임 (기준 게임·시리즈 제외)".

**콜드스타트:** 취향 분석 슬라이더 49개 전부 5.0 → 빠른 시작 프리셋 5개(힐링/전략/서사/액션/공포) 클릭 즉시 추천.
비로그인 결과에 "떠나면 사라짐 → 로그인 시 저장" 소프트 넛지 (막지 않고 잃는 걸 알려서 동기 부여).

**그 외:** 카드에 핵심 지표 2개 노출 + gem 뱃지 툴팁(매치율과 구분) / 랭킹 Top 30 + HOT 툴팁 + 빈 상태 CTA /
Vibe 칩 스크롤바 숨김·예시 칩 제거 / 푸터 하단 고정(짧은 페이지에서 중간에 뜨던 버그) /
상세 헤더 이미지 288px 캡(1080p에서 화면 점유 과다) / 히어로는 브랜드형("Hidden Gem" + 부제)으로 회귀 / 배율 0.95.

**README:** 시니어 톤 전면 재작성 — 문제→해결, mermaid 아키텍처, 증류 파이프라인, 기술 의사결정 6개, 이모지 0. `.env.example` 추가.

## 📋 다음 할 일

**데이터 (바로):**
- ⬜ 100개 재분석: `batch_generator --csv data/new_games.csv --full --yes --sync --model gpt-5.4-mini --fewshot ... --fewshot-n 12`
- ⬜ `batch_processor` → `generate_embeddings` → `recalc_percentile --yes`
- ⬜ 백필 `weekly_pipeline --from 2026-03-16 --limit 500` 반복 ("신작 없음"까지)
- ⬜ 주간 스케줄 등록 `setup_weekly_task.ps1` (리허설 통과 후)

**운영:**
- ⬜ Railway 배포 실패 알림 켜기 (11시간 다운 재발 방지)
- ⬜ Umami 운영 연결 확인 (Vercel `NEXT_PUBLIC_UMAMI_URL`) — 측정 없이 홍보 X
- ⬜ 운영에서 "위쳐 같은 게임" 검색 결과 확인 → 포트폴리오 ⑩ 결말 채우기

**메모:**
- Steam은 어필리에이트 없음 → 수익은 Humble Partner 등 서드파티 스토어, 장기적으론 B2B
- 광고는 MAU 수천 이후, 사이드 배너보다 네이티브 1칸
- 랭킹 점수 계단(100·99·90…)은 백필로 게임 늘면 완화, 그래도 거슬리면 gem_percentile 정렬로
- 커밋 메시지 한글 컨벤션 유지 (영문 커밋 2개는 히스토리 보존)